Exploration of the new data from the new CAF scraping module thing that Stoyan made to get C objects into a readable state for python tools

In [34]:
import sys
import uproot
import pandas as pd
import numpy as np
sys.path.append("/users/oz22897/atmospherics-tools/fast-osc-feedback")

from fastfeedback import *

In [37]:
data_manager = DataManager("/storage/1/st15719/caf_new_sum.2.6M_weighted.root")
data = data_manager.prepare_data()
data.head()

Loading data...


/software/oz22897/miniconda/envs/dune-atmo-osc/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:454: RuntimeWarning: invalid value encountered in cast
  return self._module.concatenate(arrays, axis=axis, casting="same_kind")


Finished loading data


run,weight,flux_nue,flux_numu,xsec,nue_w,numu_w,osc_from_e_w,osc_from_mu_w,final_oscillated_w,genie_weight,nuPDG,Ev,isCC,NuMomY,mode,recoE_numu,recoE_nue,direc_numu,direc_nue,direc_nc,cvn_numu,cvn_nue,cvn_nc,npfps,direc_true,reco_pdg,recoE,direc_reco
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f32,bool,f32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,i32,f64,f64
0,2.4045136e7,0.002161,0.007993,3.63535,0.007857,0.029058,0.999988,0.000005,0.007857,2.4045136e7,-12,35.698929,true,5.325916,1,28.529816,28.432413,-0.167463,-0.291152,-0.291152,0.001684,0.994011,0.004305,9.0,-0.14919,12,28.432413,-0.291152
0,6.0056676e7,700.832829,1458.73821,0.000058,0.040934,0.085202,0.815649,0.004269,0.033752,6.0056676e7,-12,0.425595,true,-0.055933,0,0.322256,0.208079,0.092648,0.092648,0.092648,0.001051,0.986669,0.01228,1.0,0.131423,12,0.208079,0.092648
0,5.3239364e7,15.426308,37.089519,0.001919,0.0296,0.071168,0.009378,0.936898,0.066955,5.3239364e7,-14,1.668767,true,-0.83822,1,1.20171,1.156426,0.167917,0.169637,0.169637,0.982827,0.00081,0.016363,3.0,0.502299,14,1.20171,0.167917
0,4.7361292e7,162.017806,343.329153,0.000198,0.0321,0.068023,0.021898,0.91852,0.063184,4.7361292e7,14,0.633065,true,-0.547973,0,0.5937,0.502835,0.568525,0.577708,0.577708,0.996696,0.000647,0.002657,2.0,0.865587,14,0.5937,0.577708
0,4.2285008e7,41.355044,92.242325,0.000658,0.027228,0.060732,0.00021,0.003651,0.000227,4.2285008e7,16,1.200049,false,1.047497,0,0.914734,0.299485,-0.333278,-0.333278,-0.333278,0.003598,0.002786,0.993616,1.0,-0.872879,0,0.299485,-0.333278


In [ ]:


file_path = "reco_pfp_info.root"
tree = uproot.open(file_path)["reco_pfp_tree"]

df = tree.arrays(library="pd")

print(f"entries = {len(df)}")
df.head()

entries = 2884703


,reco_ok,n_reco_pfps,n_reco_tracks,n_reco_showers,single_hits_energy,n_reco_muons_pions,n_reco_protons
0,True,10,5,4,11.48799,4,1
1,True,2,1,0,0.11094,1,0
2,True,4,3,0,0.06457,2,1
3,False,-1,-1,-1,-1.00000,-1,-1
4,True,3,2,0,0.00242,1,1


In [ ]:
import uproot
import pandas as pd
import numpy as np

MAIN_CAF = "/storage/1/st15719/caf_new_sum.2.6M_weighted.root"
PFP_FILE = "reco_pfp_info.root"

P_ENU = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.calo"
P_ELEP = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.lep_calo"

caf_tree = uproot.open(MAIN_CAF)["cafTree"]
data = caf_tree.arrays([P_ENU, P_ELEP])

initial_count = len(data[P_ENU])
print(f"Initial entries in CAF file: {initial_count}")

df_kinematics = pd.DataFrame({
    'E_nu': [x[0] if len(x) > 0 else np.nan for x in data[P_ENU]],
    'E_lep': [x[0] if len(x) > 0 else np.nan for x in data[P_ELEP]]
})

df_kinematics['hadron_energy'] = df_kinematics['E_nu'] - df_kinematics['E_lep']
df_kinematics['inelasticity'] = (df_kinematics['E_nu'] - df_kinematics['E_lep']) / df_kinematics['E_nu']

pfp_tree = uproot.open(PFP_FILE)["reco_pfp_tree"]
df_pfps = pfp_tree.arrays(library="pd")

df_final = pd.concat([df_kinematics, df_pfps], axis=1)

count_pre_filter = len(df_final)
df_final = df_final[df_final['E_nu'] > -998]
count_after_enu = len(df_final)
print(f"Entries discarded by E_nu > -998 filter: {count_pre_filter - count_after_enu}")

df_final = df_final.dropna()
count_after_dropna = len(df_final)
print(f"Entries discarded by dropna(): {count_after_enu - count_after_dropna}")

print(f"\nFinal DataFrame contains {len(df_final)} events.")
print(df_final[['E_nu', 'hadron_energy', 'inelasticity', 'n_reco_pfps']].head())

Initial entries in CAF file: 2884703
Entries discarded by E_nu > -998 filter: 233676
Entries discarded by dropna(): 0

Final DataFrame contains 2651027 events.
        E_nu  hadron_energy  inelasticity  n_reco_pfps
0  32.386795       3.856979      0.119091           10
1   0.215232      -0.107024     -0.497252            2
2   1.178813      -0.022898     -0.019424            4
4   0.507301      -0.086399     -0.170311            3
5   0.301862      -0.612872     -2.030306            2


In [38]:
df_final

,E_nu,E_lep,hadron_energy,inelasticity,reco_ok,n_reco_pfps,n_reco_tracks,n_reco_showers,single_hits_energy,n_reco_muons_pions,n_reco_protons
0,32.386795,28.529816,3.856979,0.119091,True,10,5,4,11.487990,4,1
1,0.215232,0.322256,-0.107024,-0.497252,True,2,1,0,0.110940,1,0
2,1.178813,1.201710,-0.022898,-0.019424,True,4,3,0,0.064570,2,1
4,0.507301,0.593700,-0.086399,-0.170311,True,3,2,0,0.002420,1,1
5,0.301862,0.914734,-0.612872,-2.030306,True,2,1,0,0.003319,1,0
...,...,...,...,...,...,...,...,...,...,...,...
2884698,0.158036,2.243878,-2.085843,-13.198562,True,2,1,0,0.000000,1,0
2884699,0.172791,0.260973,-0.088181,-0.510334,True,2,1,0,0.005397,1,0
2884700,0.126268,0.271612,-0.145344,-1.151073,True,3,1,1,0.012165,1,0
2884701,1.432222,1.471067,-0.038845,-0.027122,True,3,1,1,0.102556,0,1


In [ ]:
plt.plot(df_final)

In [1]:
import sys
import awkward as ak

sys.path.append("/users/xn22103/atmospherics-tools/fast-osc-feedback/")

import pandas as pd
import polars as pl
import uproot


caf_tree = uproot.open("/storage/1/st15719/caf_new_sum.2.6M_weighted.root")["cafTree"]
Enu_e_had = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.e_had"
Enu_e_calo = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.e_calo"
Enu_mu_had = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.mu_had"
Enu_mu_range = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.mu_range"
PDG = "rec/mc/mc.nu/mc.nu.pdg"
data = caf_tree.arrays([Enu_e_had, Enu_e_calo, Enu_mu_had, Enu_mu_range, PDG])

df = pl.from_pandas(ak.to_dataframe(data))
df.columns = ["Enu_e_had", "Enu_e_calo", "Enu_mu_had", "Enu_mu_range", "PDG"]

reco_pfp_tree = uproot.open("reco_pfp_info.root")["reco_pfp_tree"]
reco_pfp_df = pl.from_pandas(reco_pfp_tree.arrays(library="pd"))

data = (
    pl.concat([df,reco_pfp_df], how="horizontal")
    .with_columns(
        (1 - (pl.col("Enu_e_had") / pl.col("Enu_e_calo"))).alias("inelasticity_e")
    )
    .with_columns(
        (1 - (pl.col("Enu_mu_had") / pl.col("Enu_mu_range"))).alias("inelasticity_mu")
    )
    .with_columns((pl.col("PDG") < 0).cast(pl.Int64).alias("label"))
)

filtered = data.filter(data["reco_ok"] == True)

data = data.drop_nulls()

data

Enu_e_had,Enu_e_calo,Enu_mu_had,Enu_mu_range,PDG,reco_ok,n_reco_pfps,n_reco_tracks,n_reco_showers,single_hits_energy,n_reco_muons_pions,n_reco_protons,inelasticity_e,inelasticity_mu,label
f32,f32,f32,f32,i32,bool,i32,i32,i32,f32,i32,i32,f32,f32,i64
12.376396,28.432413,28.015844,28.529816,-12,true,10,5,4,11.48799,4,1,0.564708,0.018015,1
0.09995,0.208079,0.09995,0.322256,-12,true,2,1,0,0.11094,1,0,0.519653,0.689843,1
0.306345,1.156426,0.306345,1.20171,-14,true,4,3,0,0.06457,2,1,0.735094,0.745076,1
0.088844,0.502835,0.088844,0.5937,14,false,-1,-1,-1,-1.0,-1,-1,0.823313,0.850355,0
0.003319,0.299485,0.003319,0.299201,16,true,3,2,0,0.00242,1,1,0.988919,0.988908,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-8.1044e-17,0.156685,0.0,0.256748,14,true,4,2,1,0.163734,0,2,1.0,1.0,0
0.002552,0.169946,0.002552,0.260973,14,true,3,1,1,0.069598,0,1,0.984981,0.990219,0
0.014027,0.12414,0.014027,0.207403,14,true,3,2,0,0.001141,1,1,0.887005,0.932367,0


In [3]:
data_pd = data.to_pandas()

is_equal = data_pd.equals(df_final)

NameError: name 'df_final' is not defined